# DCGAN — Unsupervised Representation Learning with Deep Convolutional GANs (2015)

_Papers · Generative Models_

**A recipe of architecture rules — all-convolutional, batchnorm, no fully-connected — that finally made image GANs train stably.**

---

This notebook is a practice scaffold for the **DCGAN — Unsupervised Representation Learning with Deep Convolutional GANs (2015)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns**. We pull a few raw samples from the MNIST dataset to see them before any transforms.

In [ ]:
import torchvision

preview = torchvision.datasets.MNIST(root="./data", train=True, download=True)
print("dataset: MNIST   samples:", len(preview))
first_image, first_label = preview[0]
print("one sample:", first_image.size, "image,  label =", first_label)
print("classes:", getattr(preview, "classes", "(digit labels 0-9)"))
samples = [preview[i] for i in range(5)]
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

DCGAN is a set of architecture rules that made image GANs train stably: all-convolutional, BatchNorm, no fully-connected hidden layers. We build it in five steps: (1) sanity-check the transposed-conv upsampling formula, (2) the all-convolutional generator, (3) the strided-conv discriminator plus DCGAN weight init, (4) the data pipeline and the minimax training loop, and (5) an ablation that swaps in a fully-connected generator.

### Step 1 — Sanity-check the transposed-conv upsampling rule

The generator upsamples by **transposed convolution**. Its output size follows `H_out = (H_in - 1)*s - 2p + k`. With `k=4, s=2, p=1` a feature map doubles (4x4 -> 8x8), which is exactly how DCGAN climbs from a tiny spatial extent to a full image. We verify the formula against a real `ConvTranspose2d` layer.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

def convT_out(h_in, k, s, p):                 # H_out = (H_in - 1)*s - 2p + k
    return (h_in - 1) * s - 2 * p + k

print("formula  4x4 ->", convT_out(4, 4, 2, 1), "x", convT_out(4, 4, 2, 1))   # 8 x 8

probe = nn.ConvTranspose2d(8, 8, kernel_size=4, stride=2, padding=1)
probe_out = probe(torch.zeros(1, 8, 4, 4))
print("real shape:", tuple(probe_out.shape))            # (1, 8, 8, 8)
assert convT_out(4, 4, 2, 1) == 8 == probe_out.shape[-1]

### Step 2 — Build the all-convolutional generator

The generator turns a (N, 100, 1, 1) latent into a 32x32 image purely with transposed convolutions — **no fully-connected layer**. Each stage doubles the spatial size (1->4->8->16->32) while shrinking channels, with BatchNorm + ReLU on hidden layers. The **output** layer is special: no BatchNorm, and a Tanh squashes pixels into [-1, 1].

In [ ]:
# DCGAN generator (all-convolutional, BatchNorm+ReLU, Tanh output).  z: (N,100,1,1)
# We generate 32x32 (4 -> 8 -> 16 -> 32) and the loader pads MNIST-sized data to 32.
NZ, NGF, NDF, NC = 100, 64, 64, 1

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(NZ, NGF * 4, 4, 1, 0, bias=False),   # 1x1 -> 4x4  (no FC layer)
            nn.BatchNorm2d(NGF * 4), nn.ReLU(True),
            nn.ConvTranspose2d(NGF * 4, NGF * 2, 4, 2, 1, bias=False),  # 4 -> 8
            nn.BatchNorm2d(NGF * 2), nn.ReLU(True),
            nn.ConvTranspose2d(NGF * 2, NGF, 4, 2, 1, bias=False),      # 8 -> 16
            nn.BatchNorm2d(NGF), nn.ReLU(True),
            nn.ConvTranspose2d(NGF, NC, 4, 2, 1, bias=False),          # 16 -> 32
            nn.Tanh())                                                  # output: NO BatchNorm, Tanh -> [-1,1]

    def forward(self, z):
        return self.net(z)

### Step 3 — Build the discriminator and DCGAN weight init

The discriminator mirrors the generator: all **strided** convolutions (no pooling), LeakyReLU(0.2), and BatchNorm on hidden layers but **not** the input layer. It halves the spatial size each stage (32->16->8->4->1) down to a single logit. Finally we apply DCGAN's weight init (Section 4): zero-centered normal with std 0.02.

In [ ]:
# DCGAN discriminator (all-strided conv, LeakyReLU 0.2, BatchNorm on hidden layers).
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(NC, NDF, 4, 2, 1, bias=False),                   # 32 -> 16  (input layer: NO BatchNorm)
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(NDF, NDF * 2, 4, 2, 1, bias=False),              # 16 -> 8
            nn.BatchNorm2d(NDF * 2), nn.LeakyReLU(0.2, True),
            nn.Conv2d(NDF * 2, NDF * 4, 4, 2, 1, bias=False),          # 8 -> 4
            nn.BatchNorm2d(NDF * 4), nn.LeakyReLU(0.2, True),
            nn.Conv2d(NDF * 4, 1, 4, 1, 0, bias=False))                # 4 -> 1x1 logit

    def forward(self, x):
        return self.net(x).view(-1)

# DCGAN weight init: zero-centered normal, std 0.02 (Section 4).
def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)

G = Generator().to(device)
G.apply(init_weights)
D = Discriminator().to(device)
D.apply(init_weights)

### Step 4 — Load the data and train the minimax game

We scale Fashion-MNIST to [-1, 1] (matching the generator's Tanh) and pad 28 -> 32. Training alternates two Adam steps (lr=2e-4, beta1=0.5, per Section 4): the **discriminator** learns to score real as 1 and fake as 0, then the **generator** learns to fool it (push `D(fake)` toward 1). We track a fixed batch of noise so the generated-sample statistics can be compared across epochs as they drift toward the real distribution.

In [ ]:
# Fashion-MNIST scaled to [-1,1] to match Tanh, padded 28 -> 32.
tfm = T.Compose([T.Resize(32), T.ToTensor(), T.Normalize((0.5,), (0.5,))])
data = torchvision.datasets.FashionMNIST(root="./data", train=True, download=True, transform=tfm)
loader = torch.utils.data.DataLoader(torch.utils.data.Subset(data, range(8000)),
                                     batch_size=128, shuffle=True, drop_last=True)

# Train the minimax game: Adam, lr=2e-4, beta1=0.5 (Section 4).
bce = nn.BCEWithLogitsLoss()
optD = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
optG = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
fixed_z = torch.randn(64, NZ, 1, 1, device=device)   # track these same samples across training

real_mean = next(iter(loader))[0].mean().item()
print(f"real data pixel mean ~ {real_mean:.3f}  (target for generated samples)")
for epoch in range(3):
    for x, _ in loader:
        x = x.to(device)
        n = x.size(0)
        # (a) D step: real -> 1, fake -> 0
        optD.zero_grad()
        z = torch.randn(n, NZ, 1, 1, device=device)
        fake = G(z)
        lossD = bce(D(x), torch.ones(n, device=device)) + \
                bce(D(fake.detach()), torch.zeros(n, device=device))
        lossD.backward()
        optD.step()
        # (b) G step: wants D(fake) -> 1
        optG.zero_grad()
        lossG = bce(D(fake), torch.ones(n, device=device))
        lossG.backward()
        optG.step()
    with torch.no_grad():
        s = G(fixed_z)
        print(f"epoch {epoch}  lossD {lossD.item():.3f}  lossG {lossG.item():.3f}  "
              f"sample mean {s.mean().item():+.3f} std {s.std().item():.3f}")
# Generated-sample stats drift toward the real data's distribution as G improves --
# samples go from noise toward recognizable garments. (Our small run, not the paper's numbers.)

### Step 5 — Ablation: swap in a fully-connected generator

To isolate what the all-convolutional rule buys us, we define a pre-DCGAN-style generator made of dense layers that emit a flat image and reshape it. Trained with the **same** discriminator, loss, Adam settings, and data, its samples come out blurrier and less spatially coherent — convolutions' weight-sharing inductive bias is doing real work (Section 3).

In [ ]:
# ABLATION: swap the convolutional G for a fully-connected one (breaks "all-conv").
class FCGenerator(nn.Module):                 # the pre-DCGAN style: dense layers, reshape
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(NZ, 256), nn.ReLU(True),
            nn.Linear(256, 512), nn.ReLU(True),
            nn.Linear(512, NC * 32 * 32), nn.Tanh())

    def forward(self, z):
        return self.net(z.view(z.size(0), -1)).view(-1, NC, 32, 32)
# Train FCGenerator with the SAME D, loss, Adam settings, and data; compare sample sharpness.
# The convolutional generator's samples are sharper and more spatially coherent -- the
# all-convolutional guideline (Section 3) is doing real work.

## Visualize the data & results

_As DCGAN trains, do the generator's fake-sample pixel statistics drift toward the real data's, and does the convolutional generator beat a fully-connected one?_

### Step 1 — Set up the convolutional and fully-connected generators

For a head-to-head comparison we define two generators sharing the same discriminator and data: `conv_G` is the all-convolutional DCGAN generator, and `FCG` is the dense-layer baseline. We also load Fashion-MNIST and record the real data's pixel **std** (~0.66) as the target the generators should climb toward.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
import numpy as np

# Reproduces the qualitative DCGAN effect on toy data: a convolutional generator's
# samples approach the real distribution faster/closer than a fully-connected one.
torch.manual_seed(0)
NZ = 100
tfm = T.Compose([T.Resize(32), T.ToTensor(), T.Normalize((0.5,), (0.5,))])
data = torchvision.datasets.FashionMNIST(root="./data", train=True, download=True, transform=tfm)
loader = torch.utils.data.DataLoader(torch.utils.data.Subset(data, range(4000)),
                                     batch_size=128, shuffle=True, drop_last=True)
real_std = torch.cat([x for x, _ in loader]).std().item()   # ~0.66

def conv_G():
    return nn.Sequential(
        nn.ConvTranspose2d(NZ,256,4,1,0,bias=False), nn.BatchNorm2d(256), nn.ReLU(True),
        nn.ConvTranspose2d(256,128,4,2,1,bias=False), nn.BatchNorm2d(128), nn.ReLU(True),
        nn.ConvTranspose2d(128,64,4,2,1,bias=False),  nn.BatchNorm2d(64),  nn.ReLU(True),
        nn.ConvTranspose2d(64,1,4,2,1,bias=False),    nn.Tanh())

class FCG(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(NZ,256), nn.ReLU(True),
                                 nn.Linear(256,512), nn.ReLU(True),
                                 nn.Linear(512,32*32), nn.Tanh())
    def forward(self, z):
        return self.net(z.view(z.size(0),-1)).view(-1,1,32,32)

def D_net():
    return nn.Sequential(
        nn.Conv2d(1,64,4,2,1,bias=False), nn.LeakyReLU(0.2,True),
        nn.Conv2d(64,128,4,2,1,bias=False), nn.BatchNorm2d(128), nn.LeakyReLU(0.2,True),
        nn.Conv2d(128,256,4,2,1,bias=False), nn.BatchNorm2d(256), nn.LeakyReLU(0.2,True),
        nn.Conv2d(256,1,4,1,0,bias=False))

### Step 2 — Train both and compare sample std over time

The `run` helper trains a given generator against a fresh discriminator for a few hundred steps, logging the fixed-noise sample **std** at each step. Running it for both generators, the convolutional one climbs toward the real std (~0.66) while the fully-connected one lags and plateaus lower — the quantitative signature of the all-convolutional advantage.

In [ ]:
def run(make_G, steps=320):
    torch.manual_seed(0)
    G, D = make_G(), D_net()
    is_conv = not isinstance(G, FCG)
    bce = nn.BCEWithLogitsLoss()
    oD = torch.optim.Adam(D.parameters(), 2e-4, betas=(0.5,0.999))
    oG = torch.optim.Adam(G.parameters(), 2e-4, betas=(0.5,0.999))
    fixed = torch.randn(64, NZ, 1, 1)
    it = iter(loader)
    stds = []
    for t in range(steps):
        try:
            x, _ = next(it)
        except StopIteration:
            it = iter(loader)
            x, _ = next(it)
        n = x.size(0)
        z = torch.randn(n, NZ, 1, 1)
        fk = G(z if is_conv else z)
        oD.zero_grad()
        lD = bce(D(x), torch.ones(n)) + bce(D(fk.detach()), torch.zeros(n))
        lD.backward()
        oD.step()
        oG.zero_grad()
        bce(D(fk), torch.ones(n)).backward()
        oG.step()
        with torch.no_grad():
            stds.append(G(fixed).std().item())
    return stds

conv = run(conv_G)
fc = run(FCG)
idx = list(range(0,320,10))
print("real std ~", round(real_std,3))
print("conv G:", [[i, round(conv[i],3)] for i in idx])
print("fc   G:", [[i, round(fc[i],3)] for i in idx])
# Convolutional G climbs toward the real std (~0.66); the FC generator lags and plateaus lower.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a working DCGAN whose convolutional generator makes recognizable
            Fashion-MNIST samples. Replace the convolutional generator with a fully-connected one (dense
            layers that emit a flat 784-vector reshaped to 28&times;28), keeping the loss, optimizer, and
            discriminator identical, and retrain. What happens to sample quality, and which DCGAN guideline does
            this isolate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Swap only the generator: dense layers 100 &rarr; 256 &rarr; 512 &rarr; 784 with a final Tanh, reshaped to (1,28,28). Keep $D$, the loss, Adam settings, and data identical. — _An honest ablation changes exactly one thing &mdash; the generator's convolutional structure &mdash; so any quality difference is attributable to it._
- Retrain and look at the samples: the dense generator's outputs are blurrier and less spatially coherent than the convolutional generator's, often with grid-like or smeared artifacts. — _A convolution reuses the same learned filter across every location (translation-equivariant) and builds local-then-global structure; a flat dense layer must learn every pixel independently with no spatial prior._
- Conclude that the "all-convolutional / no fully-connected hidden layers" guideline is doing real work, not decoration. — _Both generators have similar parameter budgets and the same adversary; only the convolutional inductive bias differs, isolating it as the cause of sharper samples._

**Answer:** The fully-connected generator produces noticeably worse samples &mdash; blurrier and
                 less coherent &mdash; than the convolutional one, even with a comparable parameter count. Since
                 only the generator's architecture changed, this isolates the DCGAN
                 all-convolutional guideline (&sect;3) as the reason for the improvement: convolutions
                 give a spatial inductive bias (weight sharing across locations) that dense layers lack. The
                 CODEVIZ panel shows this contrast.

</details>

**Problem 2.** You want the generator's first transposed conv to lift the $z$ vector (shape $1\times1$ spatially)
            up to a $4\times4$ feature map in one step, with no fully-connected layer. Pick
            ConvTranspose2d hyperparameters and verify with the output-size formula.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the formula: $H_{\text{out}} = (H_{\text{in}} - 1)s - 2p + k$, with $H_{\text{in}} = 1$. — _The latent enters as a (N,100,1,1) tensor, so spatially $H_{\text{in}}=1$._
- Choose $k=4, s=1, p=0$ and compute: $(1-1)\times1 - 0 + 4 = 0 + 4 = 4$. — _With $H_{\text{in}}=1$ the stride term vanishes, so $H_{\text{out}} = k - 2p = 4$._
- Confirm shapes: input (N,100,1,1) &rarr; output (N, C, 4, 4) for chosen channel count $C$, no nn.Linear used. — _This realizes the "project to a small spatial extent" step of Fig. 1 purely convolutionally._

**Answer:** Use ConvTranspose2d(100, C, kernel_size=4, stride=1, padding=0). By
                 $H_{\text{out}} = (1-1)\cdot1 - 0 + 4 = 4$, the $1\times1$ latent becomes a $4\times4$ map
                 with no fully-connected layer &mdash; exactly the DCGAN generator's first step.

</details>

**Problem 3.** Your worked example used $k=4,s=2,p=1$ on a $4\times4$ input and got $8\times8$. Suppose you
            instead set padding $p=0$ (keeping $k=4,s=2$). What is the new output size, and why does that break
            the clean "double each layer" plan to reach exactly $64\times64$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Plug into the formula: $H_{\text{out}} = (4-1)\times2 - 2\times0 + 4 = 6 + 0 + 4 = 10$. — _Removing padding adds $2p = 2$ pixels back, so the output grows beyond a clean doubling._
- Note $10$ is not $8$: the layer over-expands, and chaining four such layers from $4$ gives $4\to10\to22\to46\to94$, not $64$. — _Each layer's extra two pixels compound, so the spatial path misses the target resolution._
- Restore $p=1$ to get the exact $\times2$ behavior. — _With $k=4,s=2,p=1$ the $-2p=-2$ exactly cancels the $+k$-minus-stride slack, yielding $H_{\text{out}}=2H_{\text{in}}$._

**Answer:** With $p=0$: $H_{\text{out}} = (4-1)\cdot2 - 0 + 4 = 10$, not $8$. It over-expands, so the
                 layers no longer cleanly double and the $4\to8\to16\to32\to64$ plan derails (you'd get
                 $4\to10\to22\to\dots$). The DCGAN-standard $k=4,s=2,p=1$ is chosen precisely because
                 $-2p$ cancels the slack and gives an exact $\times2$ each layer.

</details>